<a href="https://colab.research.google.com/github/gopinath5268-ui/Bike_Stations_Project_/blob/main/AQI_Analysis_FinalProject_Format.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Air Quality Analytics: A CPCB Real-Time Data Analysis Using Python**

**Business Problem**:

*   Real-time air quality monitoring stations across India generate pollutant-level readings (PM2.5, PM10, NO2, SO2, CO, NH3, OZONE) that are collected and summarized by the Central Pollution Control Board (CPCB).
*   However, comparing pollution levels, monitoring coverage, and pollutant composition across states can prove challenging without consolidated analysis.
*   This project analyzes CPCB real-time state-level air quality data to identify pollution hotspots, dominant pollutants, and monitoring-coverage gaps across India using Python-based data analytics.

**Data Source**:

* Source: Central Pollution Control Board (CPCB) Real-Time Air Quality Monitoring Network, Ministry of Environment, Forest and Climate Change (MoEFCC), Government of India.
* Location: India (state/UT level, 29 states/UTs)
* Timeline: Real-time snapshot (single date capture — 07-07-2026)
* Domain: Environmental Statistics / Air Quality / Government Monitoring Network
* Attributes: id, date, state_name, num_stations_surveyed, num_cities_surveyed, min/max/avg readings for PM2.5, PM10, NO2, SO2, CO, NH3, OZONE, overall min/max/avg across all pollutants, and state_code.

**Import Required Libraries**:

In [ ]:
# Import required libraries for data handling, statistics and visualization

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

**Load the Dataset**:

In [ ]:
# Load the CPCB state-level air quality summary CSV into a DataFrame
# (place air_quality_state_summary.csv in the same folder as this notebook)

url = "air_quality_state_summary.csv"

df = pd.read_csv(url)

FileNotFoundError: [Errno 2] No such file or directory: 'air_quality_state_summary.csv'

**Initial Exploratory Data Analysis (EDA)**:

In [ ]:
# Display the first 5 records

print("First 5 Rows of the Dataset:\n")
print(df.head())

In [ ]:
# Display the last 5 records

print("Last 5 Rows of the Dataset:\n")
print(df.tail())

In [ ]:
# Display dataset information

print("Dataset Information:\n")
df.info()

In [ ]:
# Display column names

print("Column Names:\n")
print(df.columns)

In [ ]:
# Check the number of rows and columns

print("Number of Rows and Columns:", df.shape)

In [ ]:
# Generate descriptive statistics of numerical features

print("Descriptive Statistics:\n")
print(df.describe())

In [ ]:
# Check missing values

print("Missing Values in Each Column:\n")
print(df.isnull().sum())

In [ ]:
# Check the number of duplicate records

print("Duplicate rows:", df.duplicated().sum())

**Data Cleaning, Data Transformation and Feature Engineering**:

In [ ]:
# Check the number of duplicate records

print("Number of Duplicate Records:")
print(df.duplicated().sum())

# No duplicate records were found in the dataset.

In [ ]:
# Check Missing Values

print("Missing Values:")
print(df.isnull().sum())

# Several pollutant min/max/avg columns have a small number of missing readings
# (stations that did not report that particular pollutant on the snapshot date).

In [ ]:
# Fill missing pollutant readings with the column median
# (median is used instead of mean to avoid distortion from a few very high-pollution states)

pollutant_cols = [c for c in df.columns
                  if c.startswith(("min_", "max_", "avg_"))
                  and c not in ["min_all_pollutants", "max_all_pollutants", "avg_all_pollutants"]]

for col in pollutant_cols:
    df[col] = df[col].fillna(df[col].median())

# Verify the Missing Values After Handling
print("Missing Values After Handling:")
print(df.isnull().sum())

In [ ]:
# Rename columns to a consistent underscore format (e.g. 'PM2.5' -> 'PM2_5')

df.columns = [c.replace(".", "_") for c in df.columns]

print(df.columns.tolist())

In [ ]:
# Convert Date column to datetime format

df["date"] = pd.to_datetime(df["date"], format="%d-%m-%Y")

print(df.dtypes)

In [ ]:
# Create Stations-Per-City ratio to measure monitoring density in each state

df["Stations_Per_City"] = (df["num_stations_surveyed"] / df["num_cities_surveyed"]).round(2)

In [ ]:
# Create Dominant Pollutant feature: the pollutant with the highest average reading in each state

avg_cols = ["avg_PM2_5", "avg_PM10", "avg_NO2", "avg_SO2", "avg_CO", "avg_NH3", "avg_OZONE"]
df["Dominant_Pollutant"] = df[avg_cols].idxmax(axis=1).str.replace("avg_", "", regex=False)

In [ ]:
# Create Pollution Load Category by splitting avg_all_pollutants into quartile-based bands

df["Pollution_Load_Category"] = pd.qcut(
    df["avg_all_pollutants"], q=4, labels=["Low", "Moderate", "High", "Very High"]
)

**Final EDA**:

In [ ]:
# Display the Final Shape of the Dataset

print("Final Shape of the Dataset:", df.shape)

In [ ]:
# Display the Final Dataset Information

df.info()

In [ ]:
# Display Missing Values After Data Cleaning

print("Missing Values:")
print(df.isnull().sum())

In [ ]:
# Display the Descriptive Statistics of the Final Dataset

print("Descriptive Statistics:")
print(df.describe())

In [ ]:
# Display the Updated Data Types

print(df.dtypes)

In [ ]:
# Display the First 5 Rows of the Final Dataset

print("First 5 Rows of the Final Dataset:")
print(df.head())

**Statistical analysis**

In [ ]:
# MEASURE OF CENTRAL TENDENCY

# Mean, Median and Mode of the overall pollution level (avg_all_pollutants) across states

pollution_central_tendency = pd.DataFrame({
    "Mean": [round(df["avg_all_pollutants"].mean(), 2)],
    "Median": [round(df["avg_all_pollutants"].median(), 2)],
    "Mode": [df["avg_all_pollutants"].mode().iloc[0]]
})

pollution_central_tendency

**Interpretation:**

1. Across the 29 states/UTs, the overall pollution level (avg_all_pollutants) has a mean of 24.98 and a median of 24.9, indicating a fairly symmetric distribution with no extreme skew from a handful of very high or very low states.
2. The mode (4.5) corresponds to Mizoram, the state with the single lowest recorded overall pollution level in the snapshot.
3. Because the mean and median are close to each other, the "average" Indian state on this date sits in the moderate pollution band, roughly between 21 and 30 on the combined pollutant scale.

In [ ]:
# Variance and Standard Deviation of overall pollution level

pollution_variation = pd.DataFrame({
    "Variance": [round(df["avg_all_pollutants"].var(), 2)],
    "Standard_Deviation": [round(df["avg_all_pollutants"].std(), 2)]
})

pollution_variation

**Interpretation:**

1. The overall pollution level has a standard deviation of 10.1 around a mean of 24.98, meaning most states fall between roughly 15 and 35.
2. Haryana (46.9) and Delhi (44.8) sit well above one standard deviation from the mean, marking them as clear outliers on the high-pollution side.
3. Mizoram (4.5), Arunachal Pradesh (7.0) and Sikkim (7.4) sit well below the mean, forming a distinct low-pollution cluster of north-eastern states.

In [ ]:
# Boxplot - Distribution of Average Readings Across Pollutant Types

avg_cols = ["avg_PM2_5", "avg_PM10", "avg_NO2", "avg_SO2", "avg_CO", "avg_NH3", "avg_OZONE"]

plt.figure(figsize=(12, 6))
df[avg_cols].boxplot(grid=False)
plt.title("State-wise Distribution of Average Pollutant Readings")
plt.ylabel("Average Concentration")
plt.xlabel("Pollutant")
plt.tight_layout()
plt.show()

**Interpretation:**

1. PM10 has both the highest median and the widest spread among all seven pollutants, confirming it as the most variable and generally most elevated pollutant across states.
2. PM2.5 follows a similar but slightly lower pattern, while NH3 has the lowest median and the tightest spread, indicating consistently low ammonia readings nationwide.
3. The high-value points above each box (e.g. for PM10 and PM2.5) represent genuine high-pollution states such as Haryana and Delhi rather than data errors, so they were retained rather than removed.

In [ ]:
# The outliers were retained because they represented genuine high-pollution states rather than data errors.
# Therefore, they were not removed.

**Visualizations**

***Univariate Analysis***

In [ ]:
# Bar Chart - Frequency of Dominant Pollutant Across States

dominant_counts = df["Dominant_Pollutant"].value_counts()

plt.figure(figsize=(8, 5))
sns.barplot(x=dominant_counts.values, y=dominant_counts.index, color="brown")
plt.title("Number of States Where Each Pollutant is Dominant")
plt.xlabel("Number of States")
plt.ylabel("Pollutant")
plt.tight_layout()
plt.show()

### **Interpretation**

* **Chart type:** Horizontal Barplot
* **Columns used:** Dominant_Pollutant
* **Relationship:** Pollutant ↔ Number of States Where it is Dominant

1. The 'Dominant_Pollutant' column identifies, for each state, which of the seven measured pollutants has the highest average reading.
2. PM10 is the dominant pollutant in 23 of the 29 states/UTs, making it by far the most widespread air-quality concern in India on this date.
3. PM2.5, CO, NO2 and SO2 are each dominant in only one or two states, showing that particulate matter (PM10) rather than gaseous pollutants drives most of the country's pollution profile.

In [ ]:
# Histogram Chart - Distribution of Overall Pollution Level

plt.figure(figsize=(10, 6))
sns.histplot(data=df, x="avg_all_pollutants", bins=10, color="lightblue")
plt.title("Distribution of Overall Pollution Level (avg_all_pollutants)")
plt.xlabel("Average Pollution Level")
plt.ylabel("Number of States")
plt.tight_layout()
plt.show()

### **Interpretation**

* **Chart type:** Histogram
* **Columns used:** avg_all_pollutants
* **Relationship:** Overall Pollution Level ↔ Number of States

1. The histogram shows the distribution of the combined pollution level across all 29 states/UTs.
2. Most states cluster between 15 and 30, consistent with the mean (24.98) and median (24.9) computed earlier.
3. A small number of states extend the distribution's right tail toward 40–47 (Haryana, Delhi, Himachal Pradesh, Punjab), while a similarly small cluster sits below 10 (Mizoram, Arunachal Pradesh, Sikkim).

In [ ]:
# Boxplot - Distribution of Monitoring Stations Surveyed (log scale)

plt.figure(figsize=(6, 5))
sns.boxplot(data=df, y="num_stations_surveyed", color="orange")
plt.yscale("log")
plt.title("Distribution of Monitoring Stations Surveyed per State")
plt.ylabel("Number of Stations (log scale)")
plt.tight_layout()
plt.show()

**Interpretation**

* **Chart type**: Boxplot
* **Column used**: 'num_stations_surveyed'
* **Relationship**: 'num_stations_surveyed' ↔ Monitoring Coverage

1. The box plot shows the distribution of the number of monitoring stations surveyed per state, on a log scale because of the strong right skew.
2. The median is 11 stations, but Delhi (45) and Uttar Pradesh (61) sit far above the rest, reflecting their much denser monitoring networks.
3. Many states have just 1–3 stations, highlighting a large gap in monitoring coverage between India's most and least densely instrumented states.

In [ ]:
# Pie Chart - Pollution Load Category Distribution

category_counts = df["Pollution_Load_Category"].value_counts()

plt.figure(figsize=(6, 6))
plt.pie(category_counts.values, labels=category_counts.index, autopct="%1.1f%%", startangle=90)
plt.title("Distribution of States by Pollution Load Category")
plt.show()

**Interpretation**

* **Chart type:** Pie Chart
* **Column used:** Pollution_Load_Category
* **Relationship:** Pollution Load Category ↔ Percentage Distribution

1. This pie chart shows the percentage of states falling into each quartile-based pollution load band (Low, Moderate, High, Very High).
2. Because the bands are built from quartiles, each holds roughly a quarter of the states (Low is slightly larger at 8 states due to a tie at the cut point).
3. The "Very High" band contains 7 states, including Haryana, Delhi, Punjab and Himachal Pradesh, which warrant priority attention in pollution-control planning.

***Bivariate Analysis***

In [ ]:
# Horizontal Barplot - Overall Pollution Level by State

state_pollution = df.set_index("state_name")["avg_all_pollutants"].sort_values(ascending=False)

plt.figure(figsize=(8, 10))
sns.barplot(x=state_pollution.values, y=state_pollution.index, color="skyblue")
plt.title("Overall Pollution Level by State")
plt.xlabel("Average Pollution Level")
plt.ylabel("State")
plt.tight_layout()
plt.show()

### **Interpretation**

* **Chart type:** Horizontal Barplot
* **Columns used:** state_name, avg_all_pollutants
* **Relationship:** State ↔ Overall Pollution Level

1. This horizontal bar chart ranks all 29 states/UTs by their overall pollution level from highest to lowest.
2. Haryana (46.9), Delhi (44.8) and Punjab (39.0) lead the ranking, consistent with the north-Indian pollution belt commonly reported by CPCB.
3. Mizoram (4.5), Arunachal Pradesh (7.0) and Sikkim (7.4) sit at the opposite end, reflecting the generally cleaner air quality of India's north-eastern states.

In [ ]:
# Scatter Plot with Regression Line - Monitoring Stations vs Overall Pollution Level

plt.figure(figsize=(7, 5))
sns.regplot(
    data=df,
    x="num_stations_surveyed",
    y="avg_all_pollutants",
    scatter_kws={"color": "brown", "alpha": 0.6, "s": 40},
    line_kws={"color": "blue", "linewidth": 2}
)
plt.title("Monitoring Stations Surveyed vs Overall Pollution Level")
plt.xlabel("Number of Stations Surveyed")
plt.ylabel("Average Pollution Level")
plt.tight_layout()
plt.show()

### **Interpretation**

* **Chart type:** Scatter Plot
* **Columns used:** num_stations_surveyed, avg_all_pollutants
* **Relationship:** Monitoring Coverage ↔ Overall Pollution Level

1. The scatter plot is based on all 29 states, with stations surveyed ranging from 1 to 61 and overall pollution level ranging from 4.5 to 46.9.
2. The correlation between the two variables is weak and positive (r ≈ 0.28), so states with more monitoring stations are not strongly associated with higher measured pollution.
3. This suggests that monitoring density mainly reflects a state's population/urbanization level rather than directly driving the measured pollution values.

In [ ]:
# Horizontal Barplot - Average Reading by Pollutant Type

avg_cols = ["avg_PM2_5", "avg_PM10", "avg_NO2", "avg_SO2", "avg_CO", "avg_NH3", "avg_OZONE"]
pollutant_means = df[avg_cols].mean().sort_values(ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(x=pollutant_means.values, y=pollutant_means.index, color="orange")
plt.title("Average Reading by Pollutant Type (All States)")
plt.xlabel("Average Concentration")
plt.ylabel("Pollutant")
plt.tight_layout()
plt.show()

### **Interpretation**

* **Chart type:** Horizontal Barplot
* **Columns used:** avg_PM2_5, avg_PM10, avg_NO2, avg_SO2, avg_CO, avg_NH3, avg_OZONE
* **Relationship:** Pollutant Type ↔ Nationwide Average Reading

1. This horizontal bar chart compares the nationwide average reading for each of the seven pollutants.
2. PM10 (53.11) and PM2.5 (39.97) are the two highest on average, confirming particulate matter as the dominant nationwide pollution concern.
3. NH3 (5.20) has the lowest nationwide average, over ten times smaller than PM10, showing ammonia is a minor contributor to India's overall pollution profile compared to particulates.

***Multivariate Analysis***

In [ ]:
# Correlation Heatmap of Average Pollutant Readings

avg_cols = ["avg_PM2_5", "avg_PM10", "avg_NO2", "avg_SO2", "avg_CO", "avg_NH3", "avg_OZONE"]
correlation = df[avg_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(correlation, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap of Average Pollutant Readings")
plt.tight_layout()
plt.show()

**Interpretation**

* **Chart type**: Correlation Heatmap
* **Columns used**: avg_PM2_5, avg_PM10, avg_NO2, avg_SO2, avg_CO, avg_NH3, avg_OZONE
* **Relationship**: Pollutant Readings ↔ Correlation

1. This correlation heatmap shows the relationships between the average readings of all seven pollutants.
2. PM2.5 and PM10 are strongly positively correlated (r = 0.72), confirming that both particulate pollutants tend to rise and fall together across states — as expected, since PM2.5 is a subset of PM10.
3. SO2 is the only pollutant with mostly negative correlations to the others (e.g. -0.22 with PM10), suggesting sulfur dioxide pollution follows a different regional pattern (often industrial) than the particulate/traffic-driven pollutants.

In [ ]:
# Pairplot of Key Pollutants

sns.pairplot(df[["avg_PM2_5", "avg_PM10", "avg_NO2", "avg_CO"]], diag_kind="hist")
plt.suptitle("Pairplot of Key Pollutant Averages", y=1.02)
plt.show()

**Interpretation**:

* **Chart type**: Pairplot
* **Columns used**: avg_PM2_5, avg_PM10, avg_NO2, avg_CO
* **Relationship**: Multiple Pollutant Readings ↔ Relationships and Distributions

1. This pairplot shows the pairwise relationships and individual distributions among PM2.5, PM10, NO2 and CO.
2. The PM2.5–PM10 scatter shows the clearest upward trend of any pair, reinforcing the strong correlation seen in the heatmap.
3. Most pollutant distributions are right-skewed (a long tail toward higher values), reflecting the small group of consistently high-pollution states identified earlier (Haryana, Delhi, Punjab).

In [ ]:
# Pivot Table - Average Pollutant Readings by Pollution Load Category

pivot_table = pd.pivot_table(
    df,
    values=["avg_PM2_5", "avg_PM10", "avg_NO2"],
    index="Pollution_Load_Category",
    aggfunc="mean",
    observed=True
).round(2)

pivot_table

In [ ]:
# Grouped Bar Chart from pivot table

pivot_table[["avg_PM2_5", "avg_PM10", "avg_NO2"]].plot(kind="bar", figsize=(10, 5))
plt.title("Average PM2.5, PM10 and NO2 by Pollution Load Category")
plt.xlabel("Pollution Load Category")
plt.ylabel("Average Concentration")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Interpretation**

* **Chart type:** Grouped Bar Chart
* **Columns used:** 'Pollution_Load_Category', 'avg_PM2_5', 'avg_PM10', 'avg_NO2'
* **Relationship:** Pollution Load Category ↔ Average Pollutant Readings

1. This grouped bar chart compares average PM2.5, PM10 and NO2 readings across the four Pollution_Load_Category bands.
2. All three pollutants rise steadily from the "Low" to the "Very High" category, confirming that the quartile-based Pollution_Load_Category consistently reflects underlying pollutant concentrations rather than being driven by a single pollutant.
3. PM10 shows the steepest increase across categories, reinforcing its role (identified in the univariate analysis) as the pollutant most responsible for pushing states into the higher pollution bands.